# HGNN Evaluation

Rollout comparison of trained HGNN against ground truth on test trajectories.
Autoregressive prediction: start from true initial state, predict 200 steps.

In [ ]:
import sys

import h5py
import torch

sys.path.insert(0, "..")

from evaluation._utils import (
    animate_trajectories,
    compute_single_step_metrics,
    plot_energy,
    plot_loss_distribution,
    plot_loss_vs_distance,
    plot_pos_vel_error,
    plot_rollout_mse,
    plot_trajectories,
    run_all_rollouts,
)
from models.hgnn import HGNN

CKPT_PATH = "../checkpoints/hgnn/20260417_022425/best.pt"
TEST_PATH = "../data/output/test.h5"
MODEL_NAME = "HGNN"
TRAJ_INDICES = [0, 1, 2]

device = torch.device("cpu")
ckpt = torch.load(CKPT_PATH, weights_only=False, map_location=device)
print(f"checkpoint: epoch {ckpt.epoch}, val_loss {ckpt.val_loss:.6f}")

model = HGNN(hidden_dim=64, n_layers=4)
model.load_state_dict(ckpt.model)
model.eval()
model.to(device)
print(f"model loaded: {sum(p.numel() for p in model.parameters())} params")

with h5py.File(TEST_PATH, "r") as f:
    test_traj = f["trajectories"][:]
    test_energies = f["energies"][:]

print(f"test trajectories: {test_traj.shape}")
print(f"test energies:     {test_energies.shape}")

In [ ]:
predicted = run_all_rollouts(model, test_traj, device)
print(f"predicted: {predicted.shape}")

## 1. Trajectory rollout comparison

In [ ]:
plot_trajectories(test_traj, predicted, TRAJ_INDICES, MODEL_NAME)

In [ ]:
animate_trajectories(test_traj, predicted, TRAJ_INDICES, MODEL_NAME)

## 2. Energy conservation

In [ ]:
plot_energy(test_traj, predicted, TRAJ_INDICES, MODEL_NAME)

## 3. Rollout MSE vs time step

In [ ]:
plot_rollout_mse(test_traj, predicted)

## 4. Position error vs velocity error

In [ ]:
plot_pos_vel_error(test_traj, predicted)

## 5. Single-step loss distribution

In [ ]:
sample_losses, min_distances = compute_single_step_metrics(model, TEST_PATH, device)
plot_loss_distribution(sample_losses)

## 6. Loss vs minimum pairwise distance

In [ ]:
plot_loss_vs_distance(min_distances, sample_losses)